# AI for Skeptics: Live Coding Demo
This demo showcases the key features of the AI agent system powering the AI for Skeptics course website.

In [ ]:
# --- SETUP ---
!pip install langchain langchain-google-genai langgraph tavily-python -q

import os
from pprint import pprint
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from typing import Dict, Any
from tavily import TavilyClient
from google.colab import userdata

# Load API keys from Colab Secrets
gemini_key = userdata.get('GEMINI_API_KEY')
tavily_key = userdata.get('tavily')
os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['tavily'] = tavily_key

print('Setup complete!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.6 MB/s eta 0:00:00
Setup complete!


## Demo 1: Calling GenAI Directly from Python
**Key point:** You don't need the ChatGPT website because you can call Gemini in ONE line of Python code!

In [ ]:
# Initialize Gemini in one line
llm = init_chat_model(model='google_genai:gemini-2.5-flash')

# Call the model directly, no website needed!
response = llm.invoke('What is cognitive offloading and why should faculty care about it?')
print(response.content)

**Cognitive offloading** is the act of using external aids, tools, or resources to reduce the mental burden of a task, thereby freeing up internal cognitive resources (like working memory or attention) for other activities. Essentially, it's outsourcing part of your thinking or memory to something outside your brain.

It's a common human behavior, ranging from simple acts to complex digital strategies:

*   **Simple Examples:**
    *   Writing a shopping list instead of memorizing items.
    *   Using a calendar or planner to remember appointments.
    *   Setting an alarm or reminder.
    *   Using a calculator for arithmetic.
    *   Taking notes during a lecture or meeting.
    *   Arranging your workspace so frequently used items are easily accessible.
*   **Digital/Advanced Examples:**
    *   Using search engines (Google, Bing) to recall facts or find information.
    *   Relying on spell-check and grammar-check software.
    *   Storing passwords in a password manager.
    *   U

## Demo 2: System Prompts in Code
**Key point:** You can customize AI behavior with a system prompt, just a string of text in your code!

In [ ]:
# Create an agent with a custom system prompt
AIAgent = create_agent(
    model=llm,
    system_prompt="You are a tutor at an accredited ivy league school in America. You are tasked with helping faculty with learning about AI, both in terms of the ethics of AI and the actual workings of it. Whenever you are asked a question, you make sure to give as comprehensive an answer as possible without getting bogged down in technical language or giving overly bloated answers. You are certainly a fan of AI, as in you can see its benefits. But you are also very aware of all the issues present with AI, and thus always give accurate, unbiased answers about AI."
)

WebsiteAgent = create_agent(
    model=llm,
    system_prompt="You are a systems AI designed to find verifiable and trustworthy sources for any claim. If you believe a claim is not supported very well, you will still attempt to find sources for it, but will make clear that you believe it is not well supported or that it is a controversial topic."
)

prompt0 = 'What are the cognitive risks of students using AI for assignments, and how can faculty address them in a business school setting?'
prompt1 = 'Is it true that over-reliance on AI can negatively impact student learning outcomes?'

promptAI = HumanMessage(content=prompt0)
promptWebsite = HumanMessage(content=prompt1)

response1 = AIAgent.invoke({'messages': [promptAI]})
print(response1['messages'][-1].content)

That's an excellent and timely question, especially given the rapid integration of AI into academic and professional life. While AI offers tremendous potential for enhancing learning and productivity, we must be acutely aware of the cognitive risks it poses to students, particularly in a field like business where critical thinking, strategic analysis, and clear communication are paramount.

Let's break down these cognitive risks and then discuss how faculty in a business school setting can proactively address them.

### Cognitive Risks of Students Using AI for Assignments

When students rely too heavily or uncritically on AI tools, several key cognitive skills can be underdeveloped or even atrophy:

1.  **Reduced Critical Thinking and Problem-Solving:** If AI consistently provides ready-made answers or solutions, students are deprived of the crucial struggle involved in grappling with complex problems, analyzing data, evaluating alternatives, and formulating independent judgments. This

## Demo 3: Streaming Output
**Key point:** You can replicate the AI typing effect in Python where responses stream token by token in real time!

In [ ]:
#write a prompt
ai_system_prompt = """
You are an AI tutor for the AI for Skeptics course at William & Mary's
Raymond A. Mason School of Business. When answering questions:

Topic: The cognitive and ethical concerns of AI in education
Audience: Business school faculty with varying levels of AI familiarity
Tone: Professional, clear, and unbiased
Format:
- Start with a brief summary
- Explain key concepts in plain language
- Provide practical examples relevant to a classroom setting
- End with actionable recommendations for faculty
"""

# Recreate AIAgent with structured prompt for streaming
AIAgent = create_agent(
    model=llm,
    system_prompt=ai_system_prompt
)

# Stream the response token by token
for token, metadata in AIAgent.stream(
    {'messages': [promptAI]},
    stream_mode='messages'
):
    if token.content:
        print(token.content, end='', flush=True)

The integration of AI into academic workflows presents both powerful opportunities and significant cognitive risks for students. While AI tools can enhance efficiency and provide access to vast amounts of information, their unsupervised use can inadvertently hinder the development of essential cognitive skills critical for future business leaders.

### Cognitive Risks of AI Use in Assignments

The primary cognitive risks stem from students outsourcing core mental processes to AI, rather than actively engaging with the material themselves:

1.  **Reduced Critical Thinking and Problem-Solving:** When students rely on AI to generate answers or solutions, they bypass the crucial steps of analyzing information, evaluating arguments, identifying nuances, and formulating independent judgments. This can lead to a superficial understanding and an inability to tackle novel, unstructured problems—a core competency in business.
2.  **Decreased Information Retention and Deep Learning:** Passive con

## Demo 4: Building Custom Tools with @tool
**Key point:** One decorator turns any Python function into an AI tool.

In [ ]:
tavily_client = TavilyClient(api_key=tavily_key)

#tool for websearch
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information using Tavily."""
    return tavily_client.search(query)

#tool for concept tutor
@tool
def concept_tutor(topic: str) -> str:
    """
    Explains AI concepts clearly for faculty.
    Returns a plain language explanation of the topic.
    """
    response = AIAgent.invoke(
        {'messages': [HumanMessage(content=topic)]}
    )
    return response['messages'][-1].content

#test the web search tool directly
print('Testing web_search tool:')
print(web_search.invoke('cognitive risks of AI in education research 2024'))

Testing web_search tool:
{'query': 'cognitive risks of AI in education research 2024', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.frontiersin.org/journals/psychology/articles/10.3389/fpsyg.2025.1550621/full', 'title': 'The cognitive paradox of AI in education: between enhancement ...', 'content': 'Results showed that pretesting improved retention and engagement, but prolonged AI exposure led to memory decline (Akgun and Toker, 2024). These findings', 'score': 0.7746691, 'raw_content': None}, {'url': 'https://www.the74million.org/article/four-takeaways-from-new-report-on-ais-risks-in-education/', 'title': 'Four Takeaways from New Report on AI’s Risks in Education – The 74', 'content': '## Brookings study finds AI undermines educational and social-emotional development, as well as teacher-student trust. “The risks we found are things like shortcutting learning so that you have less cognitive development,” said Rebecca Winthrop, who heads B

## Demo 5: Agent Memory
**Key point:** Agents can remember previous messages in a conversation using InMemorySaver!

In [ ]:
# Recreate agents with memory
AIAgent = create_agent(
    model=llm,
    system_prompt="You are a tutor at an accredited ivy league school in America. You are tasked with helping faculty with learning about AI, both in terms of the ethics of AI and the actual workings of it. Whenever you are asked a question, you make sure to give as comprehensive an answer as possible without getting bogged down in technical language or giving overly bloated answers. You are certainly a fan of AI, as in you can see its benefits. But you are also very aware of all the issues present with AI, and thus always give accurate, unbiased answers about AI.",
    checkpointer=InMemorySaver()  # This gives the agent memory
)

WebsiteAgent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="You are a research assistant for the AI for Skeptics course at William & Mary. For every claim you research, you MUST use the web_search tool to find real sources. You MUST return each source with: Title, URL (full link), Publisher, and why it is relevant. Never summarize without providing actual URLs. If you cannot find a real URL do not include that source.",
    checkpointer=InMemorySaver()
)

config = {'configurable': {'thread_id': '1'}}

# First message
response_mem1 = AIAgent.invoke(
    {'messages': [HumanMessage(content='What is cognitive offloading?')]},
    config
)
print('Turn 1:', response_mem1['messages'][-1].content)

# Second message
response_mem2 = AIAgent.invoke(
    {'messages': [HumanMessage(content='Can you give me a classroom example of what we just discussed?')]},
    config
)
print('\nTurn 2 (agent remembers previous message):', response_mem2['messages'][-1].content)

Turn 1: That's a great question, and it touches upon a fascinating aspect of how we interact with information and tools, especially in the age of AI.

**Cognitive offloading** refers to the process of using external tools, objects, or even other people to reduce the mental effort required to perform a task. Essentially, it's about shifting some of the cognitive burden—like remembering, calculating, or organizing—from our internal mental processes to something outside our minds.

Think of it this way: our brains have a finite capacity for working memory and attention. When we offload a task, we free up those precious internal resources, allowing us to focus on other things, think more deeply, or simply reduce mental strain.

Here are some examples to illustrate:

*   **Traditional Examples:**
    *   **Writing a to-do list:** Instead of trying to remember every task, you write it down, offloading that memory burden to a piece of paper or a digital app.
    *   **Using a calculator:** Ra

## Demo 6: Multi-Agent Orchestration
**Key point:** You can build agents that call OTHER agents as tools. This powers the AI Assistant on our website!

In [ ]:
#creating a tool that calls the AI Agent
@tool
def call_AIAgent(topic: str, content: str) -> str:
    """
    Call AIAgent to explain AI concepts or generate a quiz.
    Returns a detailed explanation with answer explanations.
    """
    response = AIAgent.invoke(
        {'messages': [HumanMessage(content=f'{topic}: {content}')]},
        config
    )
    return response['messages'][-1].content

#creating a tool that calls the Website Agent
@tool
def call_WebsiteAgent(query: str) -> str:
    """
    Call WebsiteAgent to find verifiable sources with real URLs for a claim.
    Returns research findings formatted as: Title, URL, Publisher, Relevance.
    Must include actual clickable URLs for every source returned.
    """
    response = WebsiteAgent.invoke(
        {'messages': [HumanMessage(content=f"Find real sources with URLs for: {query}")]},
        config
    )
    return response['messages'][-1].content

# OrchestratorAgent coordinates both sub-agents
# This is exactly what powers the AI Assistant on the website
orchestration_agent = create_agent(
    model=llm,
    tools=[call_AIAgent, call_WebsiteAgent],
    system_prompt="You are an AI assistant for the AI for Skeptics course at William & Mary's Raymond A. Mason School of Business. For EVERY message you receive you must ALWAYS follow these two steps in order: Step 1 — call call_AIAgent to explain the concept or answer the question. Step 2 — call call_WebsiteAgent to find supporting sources for your response. You must always complete both steps before giving a final answer. Never answer directly from your own knowledge without calling both tools first."
)

# This is exactly what happens when a faculty member uses the AI Assistant on the website
response = orchestration_agent.invoke({
    'messages': [HumanMessage(content='What are the cognitive risks of students using AI for assignments?')]
})

# Extract the text content from the response
last_message = response['messages'][-1]
response_text = last_message.content

# Handle complex response formats
if not isinstance(response_text, str):
    text_parts = []
    for part in response_text:
        if isinstance(part, dict) and 'text' in part:
            text_parts.append(part['text'])
        elif isinstance(part, str):
            text_parts.append(part)
    response_text = "".join(text_parts)

print("OrchestratorAgent response (powers the website AI Assistant):")
print(response_text)

OrchestratorAgent response (powers the website AI Assistant):
The use of AI by students for assignments carries several significant cognitive risks that can impact deep learning and skill development. These risks include:

1.  **Reduced Deep Processing and Superficial Learning:** AI can lead to students passively consuming information rather than actively engaging in analysis, synthesis, and critical evaluation. This bypasses the cognitive effort necessary for deep understanding, potentially resulting in surface-level knowledge where students can repeat information but lack true comprehension or the ability to apply it in new contexts. The "struggle" inherent in learning, which fosters resilience and problem-solving, can be circumvented by AI providing ready-made solutions.

2.  **Skill Atrophy and Underdevelopment:** Over-reliance on AI can hinder the development of crucial skills such as:
    *   **Writing and Communication:** Students may not develop their own voice, rhetorical skil